In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", None)

print("Dia 4 (parte 2) — Treinando modelos de cancelamento e gerando previsões")

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/yashdevladdha/uber-ride-analytics-dashboard/ncr_ride_bookings.csv")

df["Booking ID"] = df["Booking ID"].str.replace('"', '', regex=False)
df["Customer ID"] = df["Customer ID"].str.replace('"', '', regex=False)

df["Date"] = pd.to_datetime(df["Date"])
df["Time"] = pd.to_datetime(df["Time"], format="%H:%M:%S")

print(f"Arquivos lidos: {len(df):,} corridas no total.")

In [ ]:
df["DateTime"] = df["Date"] + (df["Time"].dt.hour.astype("timedelta64[h]"))

df["Dia_da_Semana"] = df["DateTime"].dt.day_name()
df["Hora_do_Dia"] = df["DateTime"].dt.hour

print("Features de tempo criadas: 'Dia_da_Semana' e 'Hora_do_Dia'.")

In [ ]:
TARGET = "Booking Status"

FEATURES_CLEAN = [
    "Vehicle Type",
    "Pickup Location",
    "Drop Location",
    "Dia_da_Semana",
    "Hora_do_Dia",
]

df_ml = df[FEATURES_CLEAN + [TARGET]].copy()

mapping_status_binario = {
    "Completed": 0,
    "Cancelled": 1,
    "Cancelled by Customer": 1,
    "Cancelled by Driver": 1,
    "Incomplete": 1,
}

df_ml[TARGET] = df_ml[TARGET].map(mapping_status_binario)

df_ml = df_ml.dropna(subset=[TARGET]).copy()
df_ml[TARGET] = df_ml[TARGET].astype(int)

print("\nDistribuição do TARGET (0 = Completed, 1 = Cancel/Incomplete):")
print(df_ml[TARGET].value_counts(normalize=True).round(3))

In [ ]:
cat_cols = ["Vehicle Type", "Pickup Location", "Drop Location", "Dia_da_Semana"]

for col in cat_cols:
    df_ml[col].fillna("Unknown", inplace=True)

df_ml["Hora_do_Dia"].fillna(df_ml["Hora_do_Dia"].median(), inplace=True)

print("\nChecando se ainda há NaN nas features:")
print(df_ml[FEATURES_CLEAN].isnull().sum())

In [ ]:
X = df_ml[FEATURES_CLEAN]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nShapes dos conjuntos:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test : {X_test.shape}")

In [ ]:
classe_majoritaria = y_train.value_counts().idxmax()

y_pred_baseline = np.full_like(
    y_test,
    fill_value=classe_majoritaria
)

print("\n=== BASELINE (para comparação no Dia 5) ===")
print(f"Classe majoritária (treino): {classe_majoritaria} — previsão burra: sempre essa classe.")

In [ ]:
numeric_features = ["Hora_do_Dia"]

categorical_features = [
    "Vehicle Type",
    "Pickup Location",
    "Drop Location",
    "Dia_da_Semana",
]

preprocessador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
    remainder="drop",
)

print("\nPré-processador configurado.")

In [ ]:
modelo_lr = Pipeline(
    steps=[
        ("preprocessador", preprocessador),
        (
            "classificador",
            LogisticRegression(
                random_state=42,
                solver="liblinear",
                max_iter=1000,
            ),
        ),
    ]
)

modelo_lr.fit(X_train, y_train)

y_pred_lr = modelo_lr.predict(X_test)
y_proba_lr = modelo_lr.predict_proba(X_test)[:, 1]

print("\n=== MODELO 1 — Regressão Logística ===")
print(f"Previsões geradas: {len(y_pred_lr):,} (0 = Completed, 1 = Cancel/Incomplete)")
print(f"Exemplo primeiras 10 previsões: {y_pred_lr[:10]}")

In [ ]:
modelo_rf = Pipeline(
    steps=[
        ("preprocessador", preprocessador),
        (
            "classificador",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)
y_proba_rf = modelo_rf.predict_proba(X_test)[:, 1]

print("\n=== MODELO 2 — Random Forest ===")
print(f"Previsões geradas: {len(y_pred_rf):,}")
print(f"Exemplo primeiras 10 previsões: {y_pred_rf[:10]}")

In [ ]:
amostra = pd.DataFrame({
    "real": y_test.values[:10],
    "pred_LogReg": y_pred_lr[:10],
    "pred_RF": y_pred_rf[:10],
})

print("\n=== AMOSTRA: REAL vs PREVISÕES (10 primeiras) ===")
print(amostra.to_string(index=False))
print("\n(Abaixo: várias avaliações no próprio arquivo.)")

In [ ]:
acc_base = accuracy_score(y_test, y_pred_baseline)
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_rf = accuracy_score(y_test, y_pred_rf)

p_lr = precision_score(y_test, y_pred_lr, zero_division=0)
r_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)

p_rf = precision_score(y_test, y_pred_rf, zero_division=0)
r_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)

auc_lr = roc_auc_score(y_test, y_proba_lr)
auc_rf = roc_auc_score(y_test, y_proba_rf)

print("=== AVALIAÇÃO 1: MÉTRICAS MÚLTIPLAS ===")
print(f"{'Métrica':<14} {'Baseline':>10} {'Logística':>12} {'Random Forest':>12}")
print("-" * 50)

print(f"{'Acurácia':<14} {acc_base:>10.2%} {acc_lr:>12.2%} {acc_rf:>12.2%}")
print(f"{'Precisão (1)':<14} {'-':>10} {p_lr:>12.2%} {p_rf:>12.2%}")
print(f"{'Recall (1)':<14} {'-':>10} {r_lr:>12.2%} {r_rf:>12.2%}")
print(f"{'F1 (classe 1)':<14} {'-':>10} {f1_lr:>12.2%} {f1_rf:>12.2%}")
print(f"{'AUC-ROC':<14} {'-':>10} {auc_lr:>12.2%} {auc_rf:>12.2%}")

In [ ]:
print("=== AVALIAÇÃO 2: SUPERAMOS O BASELINE? ===")

print(f"Baseline (sempre prever classe {classe_majoritaria}): acurácia = {acc_base:.2%}")
print(f"Regressão Logística: acurácia = {acc_lr:.2%}")
print(f"Random Forest: acurácia = {acc_rf:.2%}")

print("\n→ Em dados desbalanceados, compare também F1 e AUC.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, y_pred, titulo in zip(
    axes,
    [y_pred_baseline, y_pred_lr, y_pred_rf],
    ["Baseline", "Regressão Logística", "Random Forest"]
):

    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        xticklabels=["Completed", "Cancel"],
        yticklabels=["Completed", "Cancel"]
    )

    ax.set_title(titulo)
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real")

plt.suptitle("Avaliação: Matriz de confusão")
plt.tight_layout()
plt.show()

In [ ]:
print("=== AVALIAÇÃO 4: RELATÓRIO DE CLASSIFICAÇÃO (Random Forest) ===")

print(
    classification_report(
        y_test,
        y_pred_rf,
        target_names=["Completed", "Cancel/Incomplete"]
    )
)

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)

plt.figure(figsize=(7,5))

plt.plot(
    fpr_lr,
    tpr_lr,
    label=f"Logística (AUC = {auc_lr:.3f})",
    color="#2196F3",
    lw=2
)

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC = {auc_rf:.3f})",
    color="#FF5722",
    lw=2
)

plt.plot(
    [0,1],
    [0,1],
    "k--",
    label="Aleatório"
)

plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Taxa de verdadeiros positivos")
plt.title("Avaliação: Curva ROC — Discriminação entre Completed e Cancel")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
df_clf_aval = pd.DataFrame({
    "Vehicle Type": X_test["Vehicle Type"].values,
    "real": y_test.values,
    "pred": y_pred_rf
})

def met_grupo(g):
    return pd.Series({
        "n": len(g),
        "acurácia": accuracy_score(g["real"], g["pred"]),
        "recall_1": recall_score(g["real"], g["pred"], zero_division=0)
    })

equidade_clf = df_clf_aval.groupby("Vehicle Type").apply(met_grupo)

print("=== AVALIAÇÃO 6: ACURÁCIA E RECALL(Cancel) POR TIPO DE VEÍCULO ===")
print(equidade_clf.round(3).to_string())

In [ ]:
df_clf_aval = pd.DataFrame({
    "Vehicle Type": X_test["Vehicle Type"].values,
    "Dia_da_Semana": X_test["Dia_da_Semana"].values,
    "real": y_test.values,
    "pred": y_pred_rf
})

equidade_dia = (
    df_clf_aval.groupby("Dia_da_Semana")[["real", "pred"]]
    .apply(met_grupo)
)

print("=== AVALIAÇÃO 7: ACURÁCIA E RECALL(Cancel) POR DIA DA SEMANA ===")
print(equidade_dia.round(3).to_string())

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
tn, fp, fn, tp = cm_rf.ravel()

print("=== AVALIAÇÃO 8: CONTAGEM DE ERROS (Random Forest) ===")
print(f"Verdadeiros Negativos (correto Completed): {tn:,}")
print(f"Falsos Positivos (previu Cancel, era Completed): {fp:,}")
print(f"Falsos Negativos (previu Completed, cancelou): {fn:,}")
print(f"Verdadeiros Positivos (correto Cancel): {tp:,}")

print("-> Discutir: qual erro é mais caro para o negócio?")

In [ ]:
print("\nDia 4 (linear02) concluído. Modelos treinados e 8 avaliações aplicadas.")